# PneumoScan — DenseNet-121 Transfer Learning

This notebook applies **DenseNet-121** as a feature extractor for chest X-ray pneumonia classification.  
DenseNet introduced **dense connections**—every layer receives the feature maps of all preceding layers as input—maximising feature reuse, strengthening gradient flow, and substantially reducing parameter count.  
Notably, Stanford’s **CheXNet** (2017) demonstrated that a DenseNet-121 fine-tuned on chest X-rays can match or exceed radiologist-level performance at detecting pneumonia, making this architecture a natural fit for the PneumoScan project.

**Classes:** `NORMAL`, `BACTERIA`, `VIRUS` &nbsp;|&nbsp; **Input size:** 224 × 224 px

In [ ]:
import sys, os

# Make the src/ package importable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src import config, data_loader, models, train, evaluate

# Colab / environment helpers
config.setup_colab()
config.ensure_dirs()

print("Environment ready.")
print(f"Classes : {config.CLASS_NAMES}")
print(f"Image size : {config.IMAGE_SIZE}")

## 1 — Data Loading

We load the training and validation splits and compute per-class weights to compensate for the class imbalance common in medical imaging datasets.

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Load train / validation datasets
train_ds, val_ds = data_loader.load_train_val_split()

# Compute class weights from training labels
train_labels = np.concatenate([y.numpy() for _, y in train_ds], axis=0)
if train_labels.ndim > 1:          # one-hot → integer labels
    train_labels = np.argmax(train_labels, axis=1)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(config.CLASS_NAMES)),
    y=train_labels,
)
class_weights = dict(enumerate(class_weights_array))

print("Class weights:")
for idx, name in enumerate(config.CLASS_NAMES):
    print(f"  {name:>10s}: {class_weights[idx]:.4f}")

## 2 — Model Architecture

### Why DenseNet-121?

DenseNet (Densely Connected Convolutional Networks) wires every layer to every other layer in a feed-forward fashion.  
For a network with $L$ layers, there are $\frac{L(L+1)}{2}$ direct connections instead of the usual $L$. This design offers several advantages:

| Benefit | Explanation |
|---------|-------------|
| **Feature reuse** | Each layer has access to all preceding feature maps, encouraging the network to learn compact, discriminative features. |
| **Gradient flow** | Dense connections provide short paths from the loss to early layers, alleviating vanishing gradients. |
| **Parameter efficiency** | Growth rate $k$ controls channel expansion; narrow layers suffice because every feature map is shared. |

DenseNet-121 consists of **4 dense blocks** with a growth rate of $k = 32$ and totals approximately **8 M** parameters.

### CheXNet connection

Stanford’s CheXNet (Rajpurkar et al., 2017) fine-tuned a DenseNet-121 on 112,120 frontal chest X-rays (ChestX-ray14) and achieved an F1 score exceeding that of practising radiologists on pneumonia detection.  
This demonstrated that DenseNet-121’s dense feature reuse is particularly well-suited for the subtle textural patterns found in chest radiographs, making it a strong baseline for the PneumoScan three-class task.

In [ ]:
# Build DenseNet-121 with frozen ImageNet backbone
model = models.build_densenet121(freeze=True)
model.summary()

## 3 — Training

We train the classification head on top of the frozen DenseNet-121 backbone. The `train.train_model()` function manages callbacks (early stopping, learning-rate scheduling, checkpointing) internally.

In [ ]:
history = train.train_model(
    model_name="densenet121",
    train_ds=train_ds,
    val_ds=val_ds,
    class_weights=class_weights,
)

print("Training complete.")

## 4 — Evaluation

We evaluate the trained model on the held-out test set. `evaluate.evaluate_model()` produces the classification report, confusion matrix, and per-class metrics.

In [ ]:
# Load test dataset
test_ds = data_loader.load_test_data()

# Full evaluation (metrics + plots)
results = evaluate.evaluate_model(
    model=model,
    test_ds=test_ds,
    model_name="densenet121",
)

print("Test evaluation finished.")

## Summary

DenseNet-121 brings dense connectivity and aggressive feature reuse to the pneumonia classification task.  
Key take-aways:

* **Proven medical-imaging pedigree** — CheXNet established DenseNet-121 as a state-of-the-art backbone for chest X-ray analysis, and ImageNet pre-training transfers well to this domain.
* **Strong gradient propagation** — Dense connections keep gradients healthy throughout the 121-layer network, enabling effective training even with a frozen backbone.
* **Feature richness** — Because every layer has access to all prior features, the network builds highly informative representations of lung textures, opacities, and consolidation patterns.

Compare these results with the EfficientNet-B0 and MobileNetV2 notebooks to identify the best transfer-learning backbone for PneumoScan.